# STGAN geografico su PVGIS

Workflow riproducibile per preparare i dati 2005–2018, addestrare STGAN e produrre score geografici per il 2019. L'architettura segue Deng et al., *Graph Convolutional Adversarial Networks for Spatiotemporal Anomaly Detection* (TNNLS 2022, DOI `10.1109/TNNLS.2021.3136171`) e il repository ufficiale `dleyan/STGAN`, verificato al commit `20d2f6b`.

Gli script rimangono la fonte eseguibile; questo notebook documenta il protocollo, orchestra i comandi e analizza gli output. Nessuna label di anomalia viene caricata durante training, normalizzazione, scoring o selezione della soglia.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Eseguire il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.anomaly_detection.stgan import (
    REFERENCE_CONFIG,
    REFERENCE_SEED,
    build_geographical_subgraphs,
)

PVGIS_DIR = Path(os.environ.get('PVGIS_DIR', ROOT / 'data' / 'pvgis')).resolve()
WORK_ROOT = (ROOT / 'outputs' / 'pvgis_stgan').resolve()
PREPARED_DIR = WORK_ROOT / 'prepared'
MANIFEST = PREPARED_DIR / 'manifest.csv'
REFERENCE_OUTPUT = WORK_ROOT / 'paper_reference'
SMOKE_OUTPUT = WORK_ROOT / 'smoke'

RUN_PREPARATION = False
RUN_SMOKE = False
RUN_REFERENCE = False
print('Repository:', ROOT)
print('PVGIS:', PVGIS_DIR)

## Corrispondenza con paper e repository

| Elemento | Paper/repository | Implementazione PVGIS |
|---|---|---|
| Nodi | località/sensori | località PVGIS |
| Sottografo | nodo centrale + 8 vicini | identico, `K=9` |
| Pesi | `exp(-d²/σ²)`, σ deviazione standard | identico, distanze Haversine |
| Normalizzazione | min-max training in `[-1,1]` | identica e training-only |
| Recent | un'ora per PeMS | un timestamp orario |
| Trend | sette giorni | 168 timestamp orari |
| External | weekday + hour, 31 dimensioni su PeMS | identico |
| GCGRU | 2 layer, 32 unità | identico |
| LSTM/altri layer | 2 layer, 64 unità | identico |
| GAN target | reale/normal=0, generato/fake=1 | identico |
| Loss G | `500*MSE + BCE(fake, normal)` | identica |
| Loss D | media BCE reale/fake | identica |
| Score | `sG + λ*sD`, componenti normalizzate, λ=1 | identico; normalizzazione stimata sul train |
| Training | 6 epoche, batch 256, Adam 1e-3, seed 20 | identico |

Adattamenti inevitabili: PVGIS non possiede una rete stradale, quindi gli archi sono KNN geografici simmetrizzati; la divisione è 2005–2018/2019; la decisione binaria usa una soglia IQR per località stimata sul train, invece del top-K% del test usato dal paper per la valutazione con ground truth incompleta. Gli score per feature sono diagnostici aggiuntivi e non una diagnosi causale.

In [ ]:
paper_config = REFERENCE_CONFIG.to_dict()
paper_config['seed'] = REFERENCE_SEED
pd.Series(paper_config, name='valore').to_frame()

## 1. Preparazione sincronizzata

La preparazione riusa il contratto CSV di MTGFlow ma aggiunge latitudine e longitudine. Non applica la normalizzazione stagionale: STGAN esegue internamente il min-max globale calcolato esclusivamente sul training.

In [ ]:
expected = [PVGIS_DIR / f'piedmont_pvgis_{year}.nc' for year in range(2005, 2020)]
missing = [path for path in expected if not path.is_file()]
print(f'NetCDF presenti: {len(expected) - len(missing)}/{len(expected)}')
if missing:
    print('Primi file mancanti:', missing[:3])

prepare_command = [
    sys.executable, str(ROOT / 'scripts' / 'prepare_pvgis_stgan.py'),
    '--pvgis-dir', str(PVGIS_DIR),
    '--out-dir', str(PREPARED_DIR),
    '--train-start', '2005', '--train-end', '2018', '--test-year', '2019',
]
print(' '.join(prepare_command))
if RUN_PREPARATION:
    subprocess.run(prepare_command, cwd=ROOT, check=True)

In [ ]:
if MANIFEST.is_file():
    manifest = pd.read_csv(MANIFEST)
    required = {'location', 'site_key', 'train_csv', 'test_csv', 'latitude', 'longitude'}
    assert required.issubset(manifest.columns)
    print(f'Località sincronizzate: {len(manifest):,}')
    display(manifest.head())
else:
    manifest = None
    print('Manifest non trovato: abilitare RUN_PREPARATION e rieseguire la cella precedente.')

## 2. Controllo del grafo geografico

La matrice globale densa non viene usata dal modello. Per ogni località vengono conservati nove indici e una matrice locale `9×9`.

In [ ]:
if manifest is not None:
    graph = build_geographical_subgraphs(
        manifest['latitude'].to_numpy(),
        manifest['longitude'].to_numpy(),
        subgraph_size=REFERENCE_CONFIG.subgraph_size,
    )
    print('Indici:', graph.node_indices.shape)
    print('Adiacenze locali:', graph.normalized_adjacency.shape)
    print(f'Sigma paper: {graph.sigma_km:.3f} km')
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(manifest['longitude'], manifest['latitude'], s=5, color='black')
    for source in range(len(manifest)):
        for target in graph.node_indices[source, 1:]:
            ax.plot(
                manifest.loc[[source, target], 'longitude'],
                manifest.loc[[source, target], 'latitude'],
                color='steelblue', alpha=0.08, linewidth=0.5,
            )
    ax.set(title='Sottografi KNN geografici PVGIS', xlabel='Longitudine', ylabel='Latitudine')
    plt.show()

## 3. Smoke test opzionale

Questa configurazione ridotta verifica soltanto l'integrazione. Non è una configurazione scientifica del paper perché usa nove località, una sola epoca e campionamento storico.

In [ ]:
smoke_command = [
    sys.executable, str(ROOT / 'scripts' / 'run_pvgis_stgan.py'),
    '--manifest', str(MANIFEST), '--out-dir', str(SMOKE_OUTPUT),
    '--max-locations', '9', '--epochs', '1',
    '--batch-size', '32', '--train-samples-per-epoch', '512',
    '--train-score-stride', '168', '--score-stride', '24',
    '--device', 'cuda',
]
print(' '.join(smoke_command))
if RUN_SMOKE:
    subprocess.run(smoke_command, cwd=ROOT, check=True)

## 4. Run di riferimento paper-aligned

`train-samples-per-epoch=0` visita l'intero prodotto tempo×località a ogni epoca, come il repository ufficiale. È estremamente costoso su 14 anni e 1.149 località. `train-score-stride=10` riguarda soltanto la stima label-free della soglia storica, che non esiste nel protocollo top-K% del paper.

In [ ]:
reference_command = [
    sys.executable, str(ROOT / 'scripts' / 'run_pvgis_stgan.py'),
    '--manifest', str(MANIFEST), '--out-dir', str(REFERENCE_OUTPUT),
    '--epochs', '6', '--batch-size', '256', '--lr', '0.001',
    '--hidden-size', '64', '--n-layers', '2', '--subgraph-size', '9',
    '--recent-steps', '1', '--trend-steps', '168',
    '--generator-reconstruction-weight', '500',
    '--score-component-weight', '1', '--train-samples-per-epoch', '0',
    '--train-score-stride', '10', '--score-stride', '1',
    '--seed', '20', '--device', 'cuda',
]
print(' '.join(reference_command))
if RUN_REFERENCE:
    subprocess.run(reference_command, cwd=ROOT, check=True)

## 5. Analisi spaziale dei risultati

`anomaly_scores.csv` mantiene il formato comune ai detector. Le coordinate vengono aggiunte con un join a `locations.csv`, evitando di ripeterle per milioni di righe.

In [ ]:
ANALYSIS_ROOT = REFERENCE_OUTPUT
SEED = 20
score_path = ANALYSIS_ROOT / f'seed_{SEED}' / 'anomaly_scores.csv'
locations_path = ANALYSIS_ROOT / 'locations.csv'
feature_path = ANALYSIS_ROOT / f'seed_{SEED}' / 'entity_anomaly_scores.csv'

if score_path.is_file() and locations_path.is_file():
    scores = pd.read_csv(score_path, parse_dates=['timestamp'])
    locations = pd.read_csv(locations_path)
    spatial = scores.merge(locations, on='location', how='left', validate='many_to_one')
    assert spatial[['latitude', 'longitude']].notna().all().all()
    display(spatial.head())
    print(f'Punti valutati: {len(spatial):,}; anomalie: {spatial.is_anomaly.sum():,}')
else:
    scores = spatial = None
    print('Output di riferimento non ancora disponibile.')

In [ ]:
if spatial is not None:
    hourly = spatial.groupby('timestamp', as_index=False).agg(
        mean_score=('anomaly_score', 'mean'),
        anomalous_locations=('is_anomaly', 'sum'),
    )
    focus_time = hourly.nlargest(1, 'anomalous_locations').iloc[0]['timestamp']
    focus = spatial.loc[spatial['timestamp'].eq(focus_time)]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(hourly['timestamp'], hourly['anomalous_locations'], linewidth=0.8)
    axes[0].set(title='Località anomale nel tempo', ylabel='Numero località')
    points = axes[1].scatter(
        focus['longitude'], focus['latitude'], c=focus['anomaly_score'],
        s=18, cmap='magma',
    )
    axes[1].set(title=f'Score spaziale — {focus_time}', xlabel='Longitudine', ylabel='Latitudine')
    fig.colorbar(points, ax=axes[1], label='Anomaly score')
    plt.tight_layout()
    plt.show()

if feature_path.is_file():
    feature_scores = pd.read_csv(feature_path, parse_dates=['timestamp'])
    display(
        feature_scores.groupby('entity', as_index=False)['contribution_fraction']
        .mean().sort_values('contribution_fraction', ascending=False)
    )